In [ ]:
# uncomment these and run this cell if needed
!pip install evaluate
!pip install huggingface_hub

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
from transformers import get_scheduler
from transformers import Trainer, TrainingArguments
import evaluate
import numpy as np

In [ ]:
#initalize variables
header1 = "Question: "
header2 = "Passage: "
header3 = "Answer: "

column_names1 = "question"
column_names2 = "passage"
column_names3 = "answer"

def set_header1(header_value):
  global header1
  header1=header_value

def set_header2(header_value):
  global header2
  header2=header_value

def set_header3(header_value):
  global header3
  header3=header_value

def set_column_names1(column_names):
  global column_names1
  column_names1=column_names

def set_column_names2(column_names):
  global column_names2
  column_names2=column_names

def set_column_names3(column_names):
  global column_names3
  column_names3=column_names

def set_master_columns(col1, col2, col3, header_val1, header_val2, header_val3):
  global column_names1, column_names2, column_names3, header1, header2, header3
  column_names1=col1
  column_names2=col2
  column_names3=col3

  header1=header_val1
  header2=header_val2
  header3=header_val3

In [ ]:
# prepping the dataset
raw_datasets = load_dataset("flowaicom/HaluEval")#https://huggingface.co/datasets/flowaicom/HaluEval

def label_map(example):
    if example["label"] == "PASS" or example["label"] == "1" or example["label"] == 1:
        example["label"] = 1
    else:
        example["label"] = 0
    return example

raw_datasets = raw_datasets.map(label_map)

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_function(examples):
    global header1, header2, header3, column_names1, column_names2, column_names3
    combined = [
        (header1 + q, header2 + p + header3 + a)
        for q, p, a in zip(examples[column_names1], examples[column_names2], examples[column_names3])
    ]

    return tokenizer(
        [c[0] for c in combined],
        [c[1] for c in combined],
        padding="max_length",
        truncation=True
    )
#[SEP] is a special token used in BERT to separate different parts of the input sequence, such as the question, context, and answer, allowing the model to distinguish between them.
#[CLS] start of input
#tokenizer does this automatically for us



In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

#added average binary because it tells the model to Treat this as a binary classification problem and focus on the positive class (1)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=predictions, references=labels, average="binary")["recall"],
        "f1": f1.compute(predictions=predictions, references=labels, average="binary")["f1"],
    }

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
set_master_columns("passage", "question", "answer", "Passage: ", "Question: ", "Answer: ")


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# split dataset since we do not have an training data only test data
split_datasets = tokenized_datasets["test"].train_test_split(test_size=0.2, seed=42)
#train_test_split = “don't test on what you trained on”

small_train_dataset = split_datasets["train"].shuffle(seed=42).select(range(400))
small_eval_dataset = split_datasets["test"].shuffle(seed=42).select(range(100))


In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#train
trainer = Trainer(
    model=model, args=training_args, train_dataset=small_train_dataset, eval_dataset=small_eval_dataset, compute_metrics=compute_metrics,
)

trainer.evaluate()

{'eval_loss': 0.6975212693214417,
 'eval_model_preparation_time': 0.0028,
 'eval_accuracy': 0.49,
 'eval_precision': 0.49,
 'eval_recall': 1.0,
 'eval_f1': 0.6577181208053692,
 'eval_runtime': 0.8227,
 'eval_samples_per_second': 121.553,
 'eval_steps_per_second': 15.802}

In [ ]:
def tokenize_function_no_passage(examples):
    global header1, header2, header3, column_names1, column_names2, column_names3
    combined = [
        (header2 + p,  header3 + a)
        for q, p, a in zip(examples[column_names1], examples[column_names2], examples[column_names3])
    ]

    return tokenizer(
        [c[0] for c in combined],
        [c[1] for c in combined],
        padding="max_length",
        truncation=True
    )

In [ ]:
set_master_columns("passage", "question", "answer", "Passage: ", "Question: ", "Answer: ")


tokenized_datasets = raw_datasets.map(tokenize_function_no_passage, batched=True)

# split dataset since we do not have an training data only test data
split_datasets = tokenized_datasets["test"].train_test_split(test_size=0.2, seed=42)
#train_test_split = “don't test on what you trained on”

small_train_dataset = split_datasets["train"].shuffle(seed=42).select(range(400))
small_eval_dataset = split_datasets["test"].shuffle(seed=42).select(range(100))


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#train
trainer = Trainer(
    model=model, args=training_args, train_dataset=small_train_dataset, eval_dataset=small_eval_dataset, compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.182024,0.950000,0.907407,1.000000,0.951456
2,No log,0.247618,0.950000,0.907407,1.000000,0.951456
3,No log,0.201298,0.970000,0.942308,1.000000,0.970297
4,No log,0.206479,0.970000,0.942308,1.000000,0.970297
5,No log,0.218630,0.970000,0.942308,1.000000,0.970297
6,No log,0.231800,0.970000,0.942308,1.000000,0.970297
7,No log,0.237743,0.970000,0.942308,1.000000,0.970297
8,No log,0.238596,0.970000,0.942308,1.000000,0.970297


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.23859578371047974,
 'eval_accuracy': 0.97,
 'eval_precision': 0.9423076923076923,
 'eval_recall': 1.0,
 'eval_f1': 0.9702970297029703,
 'eval_runtime': 0.7654,
 'eval_samples_per_second': 130.643,
 'eval_steps_per_second': 16.984,
 'epoch': 8.0}